# Homework 3 - Lab Assignment
**Mateo Larrea - Music 320 - Fall 2025**

**Topic:** DFT Implementation

In [ ]:

# Importing needed libraries! :)
import numpy as np
import matplotlib.pyplot as plt 
%matplotlib inline

---
## Lab 1: Implement an N-long DFT as two N/2 DFT (20 points)

### Mathematical Background:
The standard DFT formula:
$$X[k]=\sum_{n=0}^{N-1}x[n] e^{-j2\pi kn/N}$$

For even N, can be decomposed as:
$$X[k]=\sum_{n=0}^{N/2-1}x[2n] e^{-j2\pi \frac{kn}{N/2}} + e^{-j2\pi \frac{k}{N}}\sum_{n=0}^{N/2-1}x[2n+1] e^{-j2\pi \frac{kn}{N/2}}$$

**Key concepts:**
- N=2 DFT returns sum and difference of two input samples
- N frequency lines from N/2 DFT obtained by repetition (DFT is periodic in frequency)

### Lab 1a: Recursive DFT Function

Write a Python function that:
- Takes N time samples (N > 2, power of two)
- Splits samples into even and odd
- Computes N/2 DFT of each by **calling the same function recursively**
- Converts N/2 frequency lines to N frequency lines
- Combines by multiplying odd sample frequency lines by twiddle factor $e^{-j2\pi \frac{k}{N}}$ and adding
- Base case: N=2 returns sum and difference samples

In [ ]:
def recursive_dft(x):

    N = len(x)  #number of samples in input signal
    if N == 2:
        return np.array([x[0] + x[1], x[0] - x[1]]) #base case DFT for N=2
    
    #split into even and odd samples
    x_even = x[::2]
    x_odd = x[1::2]
    
    #recursively compute N/2-point DFTs -- this is where the magic happens: each recursive call halves the problem size
    X_even = recursive_dft(x_even)
    X_odd = recursive_dft(x_odd) 
    
    #compute twiddle factors: W_N^k = e^(-j*2*pi*k/N)
    k = np.arange(N // 2)
    twiddle = np.exp(-2j * np.pi * k / N)
    
    #combine results using periodicity of DFT
    X = np.zeros(N, dtype=complex)  #allocate space for N-point DFT result
    X[:N//2] = X_even + twiddle * X_odd  #first half
    X[N//2:] = X_even - twiddle * X_odd  #second half
    
    return X

### Lab 1b: Test with N=32

Test the recursive DFT function with:
- Test signal: `np.sin(2* np.pi * n/4 + np.pi/8)`
- Compare real and imaginary parts with `numpy.fft.fft`

In [ ]:
#goal: verify that my recursive DFT implementation produces the same results as NumPy's highly optimized FFT implementation

#generate test signal for N=32
N = 32  # samples
n = np.arange(N) 
x_test_32 = np.sin(2 * np.pi * n / 4 + np.pi / 8)

#DFT using our recursive function!
X_recursive_32 = recursive_dft(x_test_32)
#DFT using numpy.fft.fft
X_numpy_32 = np.fft.fft(x_test_32)

#compare results
print("Comparison for N=32:")
#compare real parts
print("\nReal part comparison:")
real_diff = np.max(np.abs(X_recursive_32.real - X_numpy_32.real))
print("Max difference:", real_diff)
#compare imaginary parts
print("\nImaginary part comparison:")
imag_diff = np.max(np.abs(X_recursive_32.imag - X_numpy_32.imag))
print("Max difference:", imag_diff)
#compare overall (complex) values
print("\nOverall max difference:")
overall_diff = np.max(np.abs(X_recursive_32 - X_numpy_32))
print("Max difference:", overall_diff)

#verification that they're equal
if np.allclose(X_recursive_32, X_numpy_32):
    print("\n Recursive DFT matches numpy.fft.fft!")
else:
    print("\n Results do not match!")

### Lab 1c: Test with N=1024 and Plot

- Generate: 1024 samples of 1 kHz sine wave at Fs = 44,100 Hz
- Amplitude A normalized to 1 (full-scale)
- Apply recursive DFT function from part a)
- Convert to dBFS: `20. * np.log10( 2/N * np.abs( v ) )`
- Plot and compare with `np.fft.fft`

In [ ]:
#generate sine wave
N = 1024
Fs = 44100
f0 = 1000
A = 1.0

n = np.arange(N)
x_1khz = A * np.sin(2 * np.pi * f0 * n / Fs)
X_recursive_1024 = recursive_dft(x_1khz)
X_numpy_1024 = np.fft.fft(x_1khz)
X_recursive_dBFS = 20 * np.log10(2 / N * np.abs(X_recursive_1024) + 1e-10)
X_numpy_dBFS = 20 * np.log10(2 / N * np.abs(X_numpy_1024) + 1e-10)
freq = np.arange(N) * Fs / N

#plot comparison
plt.figure(figsize=(14, 6))
#left subplot: overlay both methods
plt.subplot(1, 2, 1)
plt.plot(freq[:N//2], X_recursive_dBFS[:N//2], 'b-', label='Recursive DFT', linewidth=2)
plt.plot(freq[:N//2], X_numpy_dBFS[:N//2], 'r--', label='numpy.fft.fft', linewidth=1.5, alpha=0.7)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dBFS)')
plt.title('DFT Comparison: Recursive vs NumPy (N=1024, 1 kHz sine)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim([-100, 10])
#right subplot: difference plot (error analysis)
plt.subplot(1, 2, 2)
plt.plot(freq[:N//2], X_recursive_dBFS[:N//2] - X_numpy_dBFS[:N//2], 'g-', linewidth=2)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Difference (dB)')
plt.title('Error: Recursive DFT - numpy.fft.fft')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Lab 1d: Timing Comparison

- Apply matrix multiplication from hw3_Lab for N=1024 to test signal from c)
- Compare timing with `np.fft.fft` implementation
- Use `%timeit` line magic in Jupyter notebook
- Note: Not worth timing recursive function (Python is slow at recursion)

In [ ]:
#matrix multiplication DFT implementation
def dft_matrix_mult(x):
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    W = np.exp(-2j * np.pi * k * n / N)
    return np.dot(W, x)

#timing comparison
print("Matrix Multiplication DFT:")
%timeit dft_matrix_mult(x_1khz)
print("\nNumPy FFT:")
%timeit np.fft.fft(x_1khz)

---
## Lab 2: Implement an N-long DFT for a real signal as an N/2 DFT (20 points)

### Mathematical Background:

**DFT Symmetries:**
- For real signal: $Z[N-k]^* = Z[k]$
- For imaginary signal: $Z[N-k]^* = -Z[k]$

**Symmetric/Antisymmetric Components:**
- Symmetric: $Z_{sym}[k] \equiv \frac{1}{2} (Z[k] + Z[N - k]^*)$ (real signal DFT)
- Antisymmetric: $Z_{antisym}[k] \equiv \frac{1}{2 j} (Z[k] - Z[N - k]^*)$ (imaginary signal DFT)

For complex signal $z[n] = x[n] + j y[n]$:
- $X[k] = Z_{sym}[k]$
- $Y[k] = Z_{antisym}[k]$

**Main Technique:**
Given real signal x[n] for n = 0, 1, ..., N-1:

Create complex samples:
$$z[n] = x[2n] + j x[2n+1]$$

Extract even and odd DFTs:
$$X_{even\_n}[k]=Z_{sym}[k] \equiv \frac{1}{2} (Z[k] + Z[N/2 - k]^*)$$
$$X_{odd\_n}[k]=Z_{antisym}[k] \equiv \frac{1}{2 j} (Z[k] - Z[N/2 - k]^*)$$

Final combination:
$$X[k]= X_{even\_n}[k]  + e^{-j2\pi \frac{k}{N}} X_{odd\_n}[k] \ \ \ \ \textrm{for } k \in 0,1,...,N-1$$

### Lab 2a: Real-Signal Optimized DFT Function

Write Python function that:
- Takes N time samples (N > 2, power of two)
- Splits into even and odd samples
- Creates length N/2 complex samples: $z[n] = x[2n] + j x[2n+1]$
- Computes length N/2 DFT of complex samples
- Computes symmetric and antisymmetric versions of N/2 frequency lines
- Uses periodicity to convert N/2 symmetric/antisymmetric to N frequency lines
- Combines with twiddle factor to get length N DFT

In [ ]:
def real_signal_dft(x):

    N = len(x)
    N_half = N // 2
    
    #split into even and odd samples and create complex signal
    #key trick: interleave even and odd samples as real and imag. parts
    x_even = x[::2]
    x_odd = x[1::2]
    z = x_even + 1j * x_odd
    #compute N/2-point DFT of complex signal 
    Z = np.fft.fft(z)
    #symmetric and antisymmetric components
    k = np.arange(N_half)
    k_reversed = np.concatenate(([0], np.arange(N_half - 1, 0, -1)))
    Z_sym = 0.5 * (Z + np.conj(Z[k_reversed]))
    Z_antisym = -0.5j * (Z - np.conj(Z[k_reversed]))
    #periodicity to extend to N frequency lines
    X_even_n = np.concatenate([Z_sym, Z_sym])
    X_odd_n = np.concatenate([Z_antisym, Z_antisym])
    #compute twiddle factors
    k_full = np.arange(N)
    twiddle = np.exp(-2j * np.pi * k_full / N)
    
    #final N-point DFT
    X = X_even_n + twiddle * X_odd_n
    
    return X

### Lab 2b: Test with N=32

Test the real-signal optimized DFT function with:
- Test signal: `np.sin(2* np.pi * n/4 + np.pi/8)`
- Compare real and imaginary parts with `numpy.fft.fft`

In [ ]:
#goal: verify that our real-signal optimized DFT produces the same results as NumPy's FFT

N = 32
n = np.arange(N)
x_test_32_lab2 = np.sin(2 * np.pi * n / 4 + np.pi / 8)

#DFT using our real-signal optimized function
X_real_opt_32 = real_signal_dft(x_test_32_lab2)
#using numpy.fft.fft
X_numpy_32_lab2 = np.fft.fft(x_test_32_lab2)

#compare results
print("Lab 2 - Comparison for N=32:")
#compare real parts
print("\nReal part comparison:")
real_diff = np.max(np.abs(X_real_opt_32.real - X_numpy_32_lab2.real))
print("Max difference:", real_diff)
#compare imaginary parts
print("\nImaginary part comparison:")
imag_diff = np.max(np.abs(X_real_opt_32.imag - X_numpy_32_lab2.imag))
print("Max difference:", imag_diff)
#compare overall (complex) values
print("\nOverall max difference:")
overall_diff = np.max(np.abs(X_real_opt_32 - X_numpy_32_lab2))
print("Max difference:", overall_diff)

#verify they are essentially equal 
if np.allclose(X_real_opt_32, X_numpy_32_lab2):
    print("\nReal-signal optimized DFT matches numpy.fft.fft!")
    print("We get the same result using only one N/2-point DFT instead of two (50% computational savings).")
else:
    print("\nResults do not match!")
    print("First 5 values comparison:")
    print("Optimized:", X_real_opt_32[:5])
    print("NumPy:    ", X_numpy_32_lab2[:5])

### Lab 2c: Test with N=1024 and Plot

- Generate: 1024 samples of 1 kHz sine wave at Fs = 44,100 Hz
- Amplitude A normalized to 1 (full-scale)
- Apply DFT function from part a)
- Convert to dBFS: `20. * np.log10( 2/N * np.abs( v ) )`
- Plot and compare with `np.fft.fft`

In [ ]:
#compute DFT using our function
X_real_opt_1024 = real_signal_dft(x_1khz)
X_numpy_1024_lab2 = np.fft.fft(x_1khz)

X_real_opt_dBFS = 20 * np.log10(2 / N * np.abs(X_real_opt_1024) + 1e-10)
X_numpy_dBFS_lab2 = 20 * np.log10(2 / N * np.abs(X_numpy_1024_lab2) + 1e-10)

freq = np.arange(N) * Fs / N

# plot comparison
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.plot(freq[:N//2], X_real_opt_dBFS[:N//2], 'b-', label='Real-signal optimized DFT', linewidth=2)
plt.plot(freq[:N//2], X_numpy_dBFS_lab2[:N//2], 'r--', label='numpy.fft.fft', linewidth=1.5, alpha=0.7)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dBFS)')
plt.title('DFT Comparison: Real-Signal Optimized vs NumPy (N=1024, 1 kHz sine)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim([-100, 10])
plt.subplot(1, 2, 2)
plt.plot(freq[:N//2], X_real_opt_dBFS[:N//2] - X_numpy_dBFS_lab2[:N//2], 'g-', linewidth=2)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Difference (dB)')
plt.title('Error: Real-Signal Optimized DFT - numpy.fft.fft')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()